# Skip-Gram exploration: understanding Word2Vec from the inside

This notebook builds two learning algorithms from ordinary NumPy operations:
full-softmax Skip-Gram and Skip-Gram with negative sampling (SGNS). The aim is
to understand what each vector, prediction, derivative, and update means.
You need basic Python, NumPy, linear algebra, derivatives, and introductory ML;
no previous NLP knowledge is assumed.

**How to study:** run cells in order; predict shapes and results before running;
pause at the questions; then change one experimental setting at a time.
The saved notebook is intentionally **unexecuted**. Numerical examples and plots
appear when you run it. Interpretation notes explain what to inspect, without
claiming a particular unobserved result.

### Setup

The core uses only **NumPy**, **Matplotlib**, and Python's standard library.
For example, in a terminal with Python installed:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install numpy matplotlib jupyterlab ipykernel
python -m jupyterlab skip_gram_exploration.ipynb
```

Select the environment's Python kernel. No notebook cell installs packages or
downloads data. Gensim is optional, disabled by default, and used only in section 18.
If you edit the corpus or vocabulary, restart the kernel and run from the top.

### Route through the notebook

1–3: text and training pairs · 4–7: network, loss, and derivatives ·
8–12: learning and inspecting vectors · 13–14: controlled experiments ·
15–18: scale, negative sampling, and practical Word2Vec · 19–20: review.


In [ ]:
import numpy as np
from collections import Counter
from time import perf_counter

np.set_printoptions(precision=4, suppress=True)
SEED = 42


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.2, "font.size": 10})


# 1. Motivation and intuition

A machine learning model needs numbers, but a word's dictionary ID is arbitrary:
ID 8 is not meaningfully twice ID 4. **Word2Vec** learns useful numerical word
representations from the contexts in which words occur, without manually labeled
semantic similarities. A **word embedding** is a learned dense vector of real
numbers, usually much shorter than the vocabulary size.

A one-hot representation has one 1 and otherwise 0s. For two different words,
their one-hot dot product is 0 and their Euclidean distance is always $\sqrt{2}$.
It therefore treats `king`–`queen` and `king`–`dog` as equally unrelated. One-hot
vectors identify words, but provide no learned notion of semantic similarity.

The **distributional hypothesis** says that words occurring in similar contexts
tend to have similar meanings or linguistic roles. This is a statistical tendency,
not a definition of meaning: antonyms can share contexts too.

Consider `the quick brown fox jumps over the lazy dog`. At `fox`, with a window
radius of 2, the context is `quick`, `brown`, `jumps`, `over`.

| Architecture | Input | What it predicts |
|---|---|---|
| CBOW (continuous bag of words) | The surrounding context words, combined | The center word, `fox` |
| Skip-Gram | The center word, `fox` | Each surrounding word in a separate pair |

Skip-Gram learns $P(\text{context}\mid\text{center})$. Predicting context is a
training task that forces the center word's vector to encode useful information
about its surroundings. It is not next-word prediction: contexts occur on both
sides, and this basic model does not encode their order or distance.
Here **center** and **target** mean the input word; the supervised label is the
context word. Some other descriptions use “target” for the label instead.


In [ ]:
fox_sentence = "the quick brown fox jumps over the lazy dog".split()
fox_position = fox_sentence.index("fox")
fox_context = fox_sentence[fox_position - 2:fox_position] + fox_sentence[fox_position + 1:fox_position + 3]
print("CBOW:", fox_context, "→ fox")
print("Skip-Gram:", [("fox", word) for word in fox_context])


**Think first:** Does Skip-Gram receive labels saying that king and queen are similar?

<details>
<summary>Reveal the answer</summary>

No. The labels are observed context words. Any useful geometry emerges through the prediction task and the shared parameters.

</details>


# 2. Creating Skip-Gram training examples

A window size of 2 means **up to two positions on each side**, not two words in
total. We clip the window at sentence boundaries, omit the center position, and
emit one ordered `(center_word, context_word)` pair per remaining position.

For `I like deep learning very much`, the center `deep` produces
`(deep, I)`, `(deep, like)`, `(deep, learning)`, `(deep, very)`.
At the left edge, `I` has only `like` and `deep` as contexts.

We omit a *position*, not every occurrence of the same word: in `go go`,
`(go, go)` is a valid pair between two different positions. Duplicates are kept
because repeated observations contribute repeatedly to the training objective.


In [ ]:
def generate_skipgram_pairs(tokens, window_size):
    if not isinstance(window_size, (int, np.integer)) or isinstance(window_size, bool) or window_size < 1:
        raise ValueError("window_size must be a positive integer")
    pairs = []
    for center_position, center_word in enumerate(tokens):
        start = max(0, center_position - window_size)
        stop = min(len(tokens), center_position + window_size + 1)
        for context_position in range(start, stop):
            if context_position != center_position:
                pairs.append((center_word, tokens[context_position]))
    return pairs


**Line by line:**

1. The function accepts one sentence's tokens and a window radius.
2. The validation rejects radii that cannot describe a positive number of positions.
3. `pairs` collects ordered observations, including repeated ones.
4. `enumerate` gives each center's position and word.
5. `max(0, ...)` keeps the left boundary inside the sentence.
6. `min(...)` clips the right boundary; `+ 1` compensates for the exclusive stop of `range`.
7. The inner loop visits every position inside those boundaries.
8. The condition excludes only the center position.
9. `append` stores the center and the current context word.
10. `return` supplies the complete list after every center has been visited.


In [ ]:
example_tokens = "I like deep learning very much".split()
example_pairs = generate_skipgram_pairs(example_tokens, window_size=2)
for position, center_word in enumerate(example_tokens):
    marked = [f"[{word}]" if index == position else word
              for index, word in enumerate(example_tokens)]
    contexts = [example_tokens[index]
                for index in range(max(0, position - 2), min(len(example_tokens), position + 3))
                if index != position]
    print(" ".join(marked))
    print("  ", [(center_word, context_word) for context_word in contexts])
print("Total pairs:", len(example_pairs))
assert len(example_pairs) == 18  # 2 + 3 + 4 + 4 + 3 + 2


**Think first:** Are (deep, learning) and (learning, deep) the same training example?

<details>
<summary>Reveal the answer</summary>

No. The first uses deep as input and learning as label; the second reverses these roles. A symmetric window usually generates both directions.

</details>


# 3. Vocabulary and encoding

Our corpus consists of nine separate short sentences. Lowercasing and splitting
on whitespace is our deliberately simple **tokenizer** (text-to-token rule).
Real tokenization must also address punctuation, casing, and language-specific rules.
We sort the unique tokens so the mapping is deterministic. IDs have no ordinal meaning.


In [ ]:
corpus = [
    "king queen royal palace",
    "man king",
    "woman queen",
    "paris france",
    "berlin germany",
    "france paris",
    "germany berlin",
    "dog cat animal",
    "cat dog animal",
]
sentences = [sentence.lower().split() for sentence in corpus]
vocabulary = sorted({word for sentence in sentences for word in sentence})
word_to_id = {word: index for index, word in enumerate(vocabulary)}
id_to_word = {index: word for word, index in word_to_id.items()}
V = len(vocabulary)
print("Vocabulary size V =", V)
print("word_to_id:", word_to_id)
print("id_to_word:", id_to_word)


In [ ]:
def one_hot(word_id, vocabulary_size):
    if not 0 <= word_id < vocabulary_size:
        raise ValueError("word_id is outside the vocabulary")
    vector = np.zeros(vocabulary_size)
    vector[word_id] = 1.0
    return vector

encoded_sentence = [word_to_id[word] for word in sentences[0]]
print("Tokens:", sentences[0])
print("Integer encoding:", encoded_sentence)
for word, word_id in zip(sentences[0], encoded_sentence):
    print(f"{word:>6} → {word_id:2} → {one_hot(word_id, V)}")


The input initially appears as one-hot because that is the algebraic representation
of a categorical word choice in the neural network. We do **not** need to store
one-hot vectors for actual training: an integer ID plus a row lookup gives exactly
the same hidden vector. The training label is also categorical: the observed context ID.


In [ ]:
def encode_training_pairs(tokenized_sentences, window_size):
    word_pairs = [pair for sentence in tokenized_sentences
                  for pair in generate_skipgram_pairs(sentence, window_size)]
    return np.array([(word_to_id[center], word_to_id[context])
                     for center, context in word_pairs], dtype=np.int64).reshape(-1, 2)

training_pairs = encode_training_pairs(sentences, window_size=2)
print("Encoded pair array shape:", training_pairs.shape)
print("First five pairs:", training_pairs[:5].tolist())


Calling the pair generator separately for each sentence prevents artificial pairs across line breaks. No padding tokens are introduced.


# 4. Skip-Gram neural network architecture

Use row-vector algebra, with NumPy storing each vector as a one-dimensional array:

| Quantity | Mathematical shape | NumPy shape | Meaning |
|---|---|---|---|
| $x$ | $1\times V$ | `(V,)` | One-hot center |
| $W_{in}$ | $V\times D$ | `(V, D)` | Input embeddings: one row per word |
| $h=xW_{in}$ | $1\times D$ | `(D,)` | Selected center embedding |
| $W_{out}$ | $D\times V$ | `(D, V)` | Output/context embeddings: one column per word |
| $z=hW_{out}$ | $1\times V$ | `(V,)` | Logits, or unnormalized scores |
| $p=\operatorname{softmax}(z)$ | $1\times V$ | `(V,)` | Context probabilities |

```text
one-hot center word → W_in → embedding vector → W_out → logits → softmax → probabilities
```

There is **no hidden activation** and no bias in this model. Two linear matrices
factor the word-to-context scores through a $D$-dimensional representation;
softmax supplies the nonlinear probability normalization.

Why does lookup work? If $x_c=1$ and every other entry is zero,
$h_j=\sum_{i=0}^{V-1}x_iW_{in,ij}=W_{in,cj}$. Every other row is multiplied by zero.


In [ ]:
toy_input_matrix = np.array([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]])
toy_x = np.array([0.0, 1.0, 0.0])
print("x shape:", toy_x.shape)
print("W_in shape:", toy_input_matrix.shape)
print("x @ W_in:", toy_x @ toy_input_matrix)
print("W_in[1]:", toy_input_matrix[1])
np.testing.assert_array_equal(toy_x @ toy_input_matrix, toy_input_matrix[1])


**Think first:** Which matrix contains the learned word embeddings?

<details>
<summary>Reveal the answer</summary>

Both matrices learn word representations: rows of W_in represent words as centers, and columns of W_out represent words as contexts. In this notebook, ‘the embeddings’ means rows of W_in. Using output embeddings or combining the two is a separate choice.

</details>


# 5. Forward propagation from scratch

For context candidate $j$, $z_j=h\cdot W_{out,:,j}$: a dot product between the
center's input embedding and candidate $j$'s output embedding.

$$p_j=\frac{\exp(z_j-m)}{\sum_k\exp(z_k-m)},\qquad m=\max_k z_k.$$

Subtracting the same constant preserves probabilities while preventing exponent
overflow. For example, scores `[1000, 1001, 1002]` produce the same probabilities
as `[0, 1, 2]`. A larger logit gets more probability, but all logits affect the
denominator: the candidates compete.


In [ ]:
def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    if logits.ndim != 1 or logits.size == 0:
        raise ValueError("softmax expects a nonempty vector")
    shifted_logits = logits - np.max(logits)
    exponentials = np.exp(shifted_logits)
    return exponentials / exponentials.sum()

print("Softmax:", softmax(np.array([1000.0, 1001.0, 1002.0])))
np.testing.assert_allclose(softmax([1000, 1001, 1002]), softmax([0, 1, 2]))


In [ ]:
def initialize_parameters(vocabulary_size, embedding_dim, seed=SEED):
    rng = np.random.default_rng(seed)
    input_embeddings = rng.normal(0.0, 0.1, (vocabulary_size, embedding_dim))
    output_weights = rng.normal(0.0, 0.1, (embedding_dim, vocabulary_size))
    return input_embeddings, output_weights

def forward(center_id, input_embeddings, output_weights):
    vocabulary_size, embedding_dim = input_embeddings.shape
    assert output_weights.shape == (embedding_dim, vocabulary_size)
    hidden = input_embeddings[center_id].copy()
    logits = hidden @ output_weights
    probabilities = softmax(logits)
    assert hidden.shape == (embedding_dim,)
    assert logits.shape == probabilities.shape == (vocabulary_size,)
    return hidden, logits, probabilities


In [ ]:
demo_W_in, demo_W_out = initialize_parameters(V, embedding_dim=3)
center_id, context_id = word_to_id["king"], word_to_id["queen"]
x = one_hot(center_id, V)
hidden, logits, probabilities = forward(center_id, demo_W_in, demo_W_out)
for name, array in [("x", x), ("W_in", demo_W_in), ("hidden", hidden),
                    ("W_out", demo_W_out), ("logits", logits), ("probabilities", probabilities)]:
    print(f"{name:>13} shape = {array.shape}")
np.testing.assert_allclose(x @ demo_W_in, hidden)
print("Hidden vector:", hidden)
print("Logits:", logits)
print("Probabilities:", probabilities)
print("Probability sum:", probabilities.sum())
assert np.isclose(probabilities.sum(), 1.0)


Inspect the nearly uniform initial probabilities. Small random weights initially give similar scores; the model has not yet seen evidence favoring one context over another.


# 6. Loss function

For one observed context $o$, the one-hot label $y$ has $y_o=1$.
The cross-entropy loss is

$$L=-\sum_{j=0}^{V-1}y_j\log p_j=-\log p_o.$$

It is a scalar (NumPy shape `()`). If the correct context has probability $0.7$,
the loss is about $0.357$; if it has probability $0.1$, the loss is about $2.303$.
We use natural logarithms, so loss is measured in **nats**.

Minimizing this loss increases the correct context's score relative to other
scores. Words with similar context distributions must solve similar prediction
problems and can acquire similar input representations. It does **not** directly
minimize input-to-input distance: `king` predicting `queen` involves king's input
vector and queen's **output** vector. This distinction will matter in the plots.


In [ ]:
manual_probabilities = np.array([0.1, 0.7, 0.2])
manual_label = np.array([0.0, 1.0, 0.0])
manual_loss = -np.sum(manual_label * np.log(manual_probabilities))
print("-log(0.7) =", -np.log(0.7))
print("One-hot cross-entropy =", manual_loss)
print("Loss if the correct probability were 0.1:", -np.log(0.1))


For training, compute the **same loss from logits** to avoid taking the logarithm
of a probability that has underflowed to zero:

$$L=\log\sum_j e^{z_j}-z_o
=\log\sum_j e^{z_j-m}-(z_o-m).$$

The function below intentionally accepts logits, not probabilities. It avoids
probability clipping, which would change the loss and its derivative.


In [ ]:
def cross_entropy_loss(logits, context_id):
    shifted_logits = np.asarray(logits, dtype=float) - np.max(logits)
    return float(np.log(np.exp(shifted_logits).sum()) - shifted_logits[context_id])

np.testing.assert_allclose(cross_entropy_loss(np.log(manual_probabilities), 1), manual_loss)
print("Loss for king → queen:", cross_entropy_loss(logits, context_id))
print("Uniform baseline log(V):", np.log(V))
print("Stable extreme-logit loss:", cross_entropy_loss(np.array([1000., -1000.]), 1))


**Think first:** Can every training pair reach probability one when a center has several different context words?

<details>
<summary>Reveal the answer</summary>

No. The same center always produces the same distribution, and its probabilities sum to one. Its repeated labels compete. The best distribution for that center matches its empirical context frequencies if the model has sufficient capacity; its average loss need not be zero.

</details>


# 7. Backpropagation by hand

We differentiate **one pair's** loss. Let $c$ be the center ID and $o$ the context
ID. All gradients use the parameter values from the forward pass.

### Step 1: probabilities → logits

The softmax Jacobian is

$$\frac{\partial p_j}{\partial z_k}=p_j(\delta_{jk}-p_k),$$

where $\delta_{jk}$ is 1 if $j=k$ and 0 otherwise. Also
$\partial L/\partial p_j=-y_j/p_j$. Applying the chain rule,

$$\frac{\partial L}{\partial z_k}
=\sum_j \frac{-y_j}{p_j}p_j(\delta_{jk}-p_k)
=-y_k+p_k\sum_j y_j=p_k-y_k.$$

Thus $g_z=p-y$ has shape `(V,)`. For $p=[0.1,0.7,0.2]$ and
$y=[0,1,0]$, $g_z=[0.1,-0.3,0.2]$. Gradient descent pushes the correct
logit upward and the others downward **if logits themselves were free parameters**.
Actual weight updates interact through the shared factorization.


In [ ]:
example_gradient_logits = manual_probabilities - manual_label
print("p - y:", example_gradient_logits)
print("Shape:", example_gradient_logits.shape)
print("Gradient entries sum to:", example_gradient_logits.sum())


### Step 2: logits → output matrix

Because $z_j=\sum_a h_aW_{out,aj}$,

$$\frac{\partial L}{\partial W_{out,aj}}=h_a g_{z,j},\qquad
G_{out}=h^Tg_z.$$

This is an **outer product**, shape `(D, V)`, implemented as `np.outer(hidden,
gradient_logits)`. Every output column generally receives a gradient in full softmax.

### Step 3: logits → hidden vector

Each $h_a$ influences every logit, so we sum those paths:

$$g_{h,a}=\sum_j g_{z,j}W_{out,aj},\qquad g_h=g_zW_{out}^T.$$

The operation is `(V,) @ (V, D) → (D,)`. It blends output vectors according to
their prediction errors: too much probability and too little probability have
opposite signs.

### Step 4: hidden vector → input matrix

Since $h_a=\sum_i x_iW_{in,ia}$,

$$G_{in,ia}=x_i g_{h,a},\qquad G_{in}=x^Tg_h.$$

This outer product has shape `(V, D)`. Only row $c$ is nonzero, because only
$x_c=1$. All other input words are unchanged by this particular pair.


In [ ]:
def backward(center_id, context_id, hidden, probabilities, output_weights):
    vocabulary_size = probabilities.size
    gradient_logits = probabilities.copy()
    gradient_logits[context_id] -= 1.0
    gradient_output = np.outer(hidden, gradient_logits)
    gradient_hidden = gradient_logits @ output_weights.T
    gradient_input = np.zeros((vocabulary_size, hidden.size))
    gradient_input[center_id] = gradient_hidden
    assert gradient_output.shape == output_weights.shape
    assert gradient_hidden.shape == hidden.shape
    return gradient_input, gradient_output, gradient_hidden, gradient_logits


In [ ]:
gradient_input, gradient_output, gradient_hidden, gradient_logits = backward(
    center_id, context_id, hidden, probabilities, demo_W_out)
for name, gradient in [("logits", gradient_logits), ("W_out", gradient_output),
                       ("hidden", gradient_hidden), ("W_in", gradient_input)]:
    print(f"Gradient {name:>6}: shape {gradient.shape}")
print("Gradient of the hidden vector:", gradient_hidden)
print("Gradient of output column queen:", gradient_output[:, context_id])
print("Nonzero input rows:", np.flatnonzero(np.any(gradient_input != 0, axis=1)))
np.testing.assert_allclose(gradient_input, np.outer(x, gradient_hidden))


### A complete numerical gradient example

With $h=[0.2,-0.1]$, the $2\times3$ output matrix below, and context ID 1,
the logits are `[0.01, -0.05, 0.0]`. Follow the values from the three prediction
errors into each output column and then back into the two hidden coordinates.


In [ ]:
tiny_hidden = np.array([0.2, -0.1])
tiny_output = np.array([[0.1, -0.2, 0.3], [0.1, 0.1, 0.6]])
tiny_probabilities = softmax(tiny_hidden @ tiny_output)
tiny_grad_in, tiny_grad_out, tiny_grad_hidden, tiny_grad_logits = backward(
    0, 1, tiny_hidden, tiny_probabilities, tiny_output)
print("Logits:", tiny_hidden @ tiny_output)
print("Probabilities:", tiny_probabilities)
print("p - y:", tiny_grad_logits)
print("outer(h, p-y):\n", tiny_grad_out)
print("(p-y) @ W_out.T:", tiny_grad_hidden)
print("Input gradient (only row 0 changes):\n", tiny_grad_in)


### Check the derivatives without automatic differentiation

For a scalar parameter $\theta$, a central finite difference estimates

$$\frac{\partial L}{\partial\theta}\approx
\frac{L(\theta+\varepsilon)-L(\theta-\varepsilon)}{2\varepsilon}.$$

We perturb each entry of tiny matrices with $\varepsilon=10^{-5}$. This is slow
but independent of our analytic derivation. It is a diagnostic, not a training method.


In [ ]:
def finite_difference_gradient(loss_function, parameter, epsilon=1e-5):
    numerical_gradient = np.zeros_like(parameter)
    for index in np.ndindex(parameter.shape):
        original_value = parameter[index]
        try:
            parameter[index] = original_value + epsilon
            loss_plus = loss_function()
            parameter[index] = original_value - epsilon
            loss_minus = loss_function()
        finally:
            parameter[index] = original_value
        numerical_gradient[index] = (loss_plus - loss_minus) / (2 * epsilon)
    return numerical_gradient


In [ ]:
check_input, check_output = initialize_parameters(4, 2, seed=7)
check_hidden, check_logits, check_probabilities = forward(1, check_input, check_output)
analytic_input, analytic_output, _, _ = backward(1, 2, check_hidden, check_probabilities, check_output)

def check_softmax_loss():
    return cross_entropy_loss(forward(1, check_input, check_output)[1], 2)

for parameter, analytic in [(check_input, analytic_input), (check_output, analytic_output)]:
    numerical = finite_difference_gradient(check_softmax_loss, parameter)
    print("Maximum absolute gradient error:", np.max(np.abs(analytic - numerical)))
    np.testing.assert_allclose(analytic, numerical, rtol=1e-5, atol=1e-7)


**Think first:** Why must the hidden gradient use the old output matrix?

<details>
<summary>Reveal the answer</summary>

Backpropagation differentiates the loss computed in the forward pass. Updating W_out first would use different parameters for part of that derivative. Compute every gradient first, then apply both updates.

</details>


# 8. Training Skip-Gram from scratch

For learning rate $\eta$, gradient descent updates
$W_{in}\leftarrow W_{in}-\eta G_{in}$ and
$W_{out}\leftarrow W_{out}-\eta G_{out}$.
**Stochastic gradient descent (SGD)** uses one observed pair per update.
An **epoch** visits every pair once, in a newly shuffled order.

We reuse the nine-line corpus from section 3. All main settings are collected
here. The weights start small and random; independent reproducible random
generators control initialization and pair ordering.


In [ ]:
WINDOW_SIZE = 2
EMBEDDING_DIM = 5
LEARNING_RATE = 0.05
EPOCHS = 300
SELECTED_WORD_PAIRS = [("king", "queen"), ("king", "dog"),
                       ("paris", "france"), ("paris", "berlin"), ("dog", "cat")]
training_pairs = encode_training_pairs(sentences, WINDOW_SIZE)
print(f"{len(sentences)} sentences, {V} words, {len(training_pairs)} training pairs")
print(f"D={EMBEDDING_DIM}, learning rate={LEARNING_RATE}, epochs={EPOCHS}, seed={SEED}")


We evaluate mean loss over **all pairs using one fixed parameter state** at each
epoch boundary. This makes the curve easier to interpret than averaging losses
measured while the weights are changing within an epoch.

$$\overline L=\frac{1}{N}\sum_{n=1}^{N}
-\log P(o_n\mid c_n).$$

The next helper returns one scalar. Repeated pairs appear repeatedly in this sum.


In [ ]:
def mean_softmax_loss(pairs, input_embeddings, output_weights):
    if len(pairs) == 0:
        raise ValueError("Training requires at least one center-context pair")
    losses = [cross_entropy_loss(forward(center, input_embeddings, output_weights)[1], context)
              for center, context in pairs]
    return float(np.mean(losses))


In [ ]:
def train_softmax(pairs, vocabulary_size, embedding_dim=5, epochs=300,
                  learning_rate=0.05, seed=SEED):
    input_embeddings, output_weights = initialize_parameters(vocabulary_size, embedding_dim, seed)
    initial_input, initial_output = input_embeddings.copy(), output_weights.copy()
    shuffle_rng = np.random.default_rng(seed + 1)
    losses = [mean_softmax_loss(pairs, input_embeddings, output_weights)]
    embedding_history = [input_embeddings.copy()]
    start_time = perf_counter()
    for epoch in range(epochs):
        for center, context in pairs[shuffle_rng.permutation(len(pairs))]:
            hidden, logits, probabilities = forward(center, input_embeddings, output_weights)
            gradient_input, gradient_output, _, _ = backward(
                center, context, hidden, probabilities, output_weights)
            input_embeddings -= learning_rate * gradient_input
            output_weights -= learning_rate * gradient_output
        losses.append(mean_softmax_loss(pairs, input_embeddings, output_weights))
        embedding_history.append(input_embeddings.copy())
    return {"W_in": input_embeddings, "W_out": output_weights,
            "initial_W_in": initial_input, "initial_W_out": initial_output,
            "losses": np.array(losses), "embedding_history": np.array(embedding_history),
            "elapsed_seconds": perf_counter() - start_time}


Read the inner loop as **lookup → predict → differentiate → update**.
`backward` forms dense gradient matrices for teaching clarity, although only one
input row changes. The snapshots are copies, so later updates cannot overwrite
our saved initial vectors. Saving every epoch is inexpensive for this tiny
corpus; large models would use occasional checkpoints instead.


In [ ]:
model = train_softmax(training_pairs, V, EMBEDDING_DIM, EPOCHS, LEARNING_RATE, SEED)
W_in, W_out = model["W_in"], model["W_out"]
print(f"Initial mean loss: {model['losses'][0]:.4f}")
print(f"Final mean loss:   {model['losses'][-1]:.4f}")
print(f"Training + evaluation time: {model['elapsed_seconds']:.2f} seconds")
assert np.isfinite(W_in).all() and np.isfinite(W_out).all()


In [ ]:
fig, ax = plt.subplots()
ax.plot(model["losses"], label="Mean training cross-entropy")
ax.axhline(np.log(V), color="gray", linestyle="--", label="Uniform prediction: log(V)")
ax.set(xlabel="Completed epochs (0 = initialization)", ylabel="Loss per pair (nats)",
       title="Learning to predict observed contexts")
ax.legend()
plt.show()


Look for an overall decrease, followed by slower improvement. Individual epochs
need not decrease monotonically with a fixed learning rate. A rising or nonfinite
loss is a reason to inspect the gradients and reduce the learning rate.

Zero is generally not attainable: `king` has several correct contexts across
different observations, yet our model outputs only one distribution for `king`.
Training loss measures fit to this corpus, **not** semantic quality on unseen text.


In [ ]:
def empirical_context_entropy(pairs):
    counts = np.zeros((V, V), dtype=float)
    np.add.at(counts, (pairs[:, 0], pairs[:, 1]), 1)
    center_counts = counts.sum(axis=1)
    center_indices, context_indices = np.nonzero(counts)
    observed_counts = counts[center_indices, context_indices]
    return float(-np.sum(observed_counts * np.log(observed_counts / center_counts[center_indices])) / len(pairs))

entropy_floor = empirical_context_entropy(training_pairs)
print("Best possible mean loss with unrestricted center distributions:", entropy_floor)
print("Our factorized model's final mean loss:", model["losses"][-1])


This empirical conditional entropy is a lower bound for the training objective:
$-\frac1N\sum_{c,o}n_{co}\log(n_{co}/n_c)$. Here $n_{co}$ counts a pair and $n_c$
counts all pairs centered on $c$. The count table has shape `(V, V)` and is used
only for this small diagnostic. A finite-dimensional factorization and finite
training need not attain the bound; zero-probability outcomes also require limiting logits.


# 9. Inspect the learned embeddings

The learned input embedding of word $w$ is $W_{in}[\mathrm{id}(w),:]$, a vector
with shape `(D,)`. Coordinates jointly encode information useful for context
prediction; a coordinate is not necessarily a named property such as “royalty.”
Changing the basis can change individual coordinates without destroying useful
relationships. We return a copy to protect the trained matrix from accidental edits.


In [ ]:
def get_embedding(word):
    if word not in word_to_id:
        raise KeyError(f"Unknown word {word!r}; choose a word from vocabulary")
    return W_in[word_to_id[word]].copy()

for word in ["king", "queen", "paris", "france", "dog", "cat"]:
    print(f"{word:>6}: {get_embedding(word)}")
print("Embedding shape:", get_embedding("king").shape)


Inspect the signs and magnitudes, but resist assigning a meaning to each coordinate. To compare words, we next measure the direction of their full vectors. The helper above always refers to the baseline model; experiment models will be kept separately.


# 10. Cosine similarity

For nonzero vectors $a,b\in\mathbb R^D$,

$$\cos(a,b)=\frac{a\cdot b}{\|a\|_2\|b\|_2},\qquad
\|a\|_2=\sqrt{\sum_{j=1}^{D}a_j^2}.$$

The dot product and both norms are scalars. The result measures **direction**:
1 means aligned, 0 means orthogonal, and −1 means opposite. Multiplying one
vector by a positive scalar leaves cosine unchanged. Cosine is not a probability.
For a zero vector the mathematical expression is undefined; our helper returns
0 as an explicit practical convention.


In [ ]:
def cosine_similarity(vector_a, vector_b):
    vector_a, vector_b = np.asarray(vector_a), np.asarray(vector_b)
    if vector_a.ndim != 1 or vector_a.shape != vector_b.shape:
        raise ValueError("Cosine similarity requires equal-length vectors")
    norm_a = np.sqrt(np.sum(vector_a ** 2))
    norm_b = np.sqrt(np.sum(vector_b ** 2))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.clip(np.dot(vector_a, vector_b) / (norm_a * norm_b), -1.0, 1.0))

print("Same direction:", cosine_similarity(np.array([1., 2.]), np.array([2., 4.])))
print("Perpendicular:", cosine_similarity(np.array([1., 0.]), np.array([0., 1.])))
for first, second in SELECTED_WORD_PAIRS:
    print(f"{first:>6} vs {second:<6}: {cosine_similarity(get_embedding(first), get_embedding(second)):+.4f}")


In [ ]:
def most_similar_in_matrix(word, embeddings, top_k=5):
    if word not in word_to_id:
        raise KeyError(f"Unknown word: {word!r}")
    if not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k must be a positive integer")
    query_embedding = embeddings[word_to_id[word]]
    scores = [(candidate, cosine_similarity(query_embedding, embeddings[word_to_id[candidate]]))
              for candidate in vocabulary if candidate != word]
    return sorted(scores, key=lambda item: (-item[1], item[0]))[:top_k]

def most_similar(word, top_k=5):
    return most_similar_in_matrix(word, W_in, top_k)

for query in ["king", "paris", "dog"]:
    print(query, "→", most_similar(query))


Inspect whether neighbors share **observed contexts**, not just a real-world category.
`paris` and `france` occur together, but their context distributions here have
different support: Paris predicts France and France predicts Paris. `paris` and
`berlin` are both capitals in our knowledge, but this corpus contains no common
`capital` context to teach that fact. `dog` and `cat` do share `animal`.

With only a few observations, random initialization, dimension, and optimization
can substantially affect similarities. Missing semantic knowledge is a limitation
of the data, not proof of a broken gradient. Even excellent prediction does not
uniquely determine input-vector cosine geometry: invertible transformations of
input vectors with compensating output transformations can preserve all logits.


**Think first:** Why might two semantically related words have high cosine similarity?

<details>
<summary>Reveal the answer</summary>

If they occur in similar contexts, training can shape their input vectors to make similar predictions. This often produces aligned vectors, but it is an emergent tendency rather than a constraint in the loss, and the corpus must supply useful evidence.

</details>


# 11. Visualizing embedding space

Our baseline vectors have five coordinates; a flat scatter plot has only two.
**Principal component analysis (PCA)** selects orthogonal directions retaining
as much centered variance as possible. It is a projection, not another embedding
training algorithm.

For embeddings $E$ of shape `(V, D)`, center each column:
$E_c=E-\mathrm{mean}(E,\mathrm{axis}=0)$. A singular value decomposition is
$E_c=U\Sigma Q^T$. The first two columns of $Q$ form a `(D, 2)` basis, so
$E_{2D}=E_cQ_{:,0:2}$ has shape `(V, 2)`.
Squared singular values are proportional to variance along each component.


In [ ]:
def pca_2d(embeddings):
    embeddings = np.asarray(embeddings, dtype=float)
    if embeddings.ndim != 2 or min(embeddings.shape) < 2:
        raise ValueError("PCA plot needs at least two words and two dimensions")
    centered = embeddings - embeddings.mean(axis=0, keepdims=True)
    _, singular_values, principal_axes = np.linalg.svd(centered, full_matrices=False)
    basis = principal_axes[:2].T
    coordinates = centered @ basis
    total_variance = np.sum(singular_values ** 2)
    retained = float(np.sum(singular_values[:2] ** 2) / total_variance) if total_variance else 0.0
    assert basis.shape == (embeddings.shape[1], 2)
    assert coordinates.shape == (embeddings.shape[0], 2)
    return coordinates, retained

toy_points = np.array([[1., 0., 0.], [2., 0., 0.], [0., 1., 0.], [0., 2., 0.]])
toy_projection, toy_retained = pca_2d(toy_points)
print("Example input shape:", toy_points.shape, "→ projected shape:", toy_projection.shape)
print("Projected coordinates:\n", toy_projection)
print("Variance retained:", toy_retained)


In [ ]:
def plot_embedding_matrix(embeddings, ax, title):
    coordinates, retained = pca_2d(embeddings)
    ax.scatter(coordinates[:, 0], coordinates[:, 1], s=35, color="tab:blue")
    for word, (horizontal, vertical) in zip(vocabulary, coordinates):
        ax.annotate(word, (horizontal, vertical), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set(xlabel="Principal component 1", ylabel="Principal component 2",
           title=f"{title}\nTwo components retain {retained:.1%} of variance")
    ax.margins(0.2)

fig, ax = plt.subplots(figsize=(9, 6))
plot_embedding_matrix(W_in, ax, "Full-softmax input embeddings")
fig.tight_layout()
plt.show()


Look for groups such as animal words or royal words, then check their original
cosine similarities rather than relying on the picture alone. Projection discards
directions: distant points can look close, and apparent clusters can be misleading.
PCA preserves centered variance, **not cosine similarity**. The axes have no
inherent semantic labels, and either axis can flip sign across runs.
For $D=2$, keeping both components loses no centered variance, but centering still
changes the origin used for cosine measurements.


# 12. Examine what training actually changes

The snapshots stored in section 8 let us compare the same coordinates before
and after training: $\Delta E=E_{\mathrm{after}}-E_{\mathrm{before}}$, shape `(V, D)`.
Large movement means the optimization changed a representation substantially;
it does not by itself mean that representation became semantically better.


In [ ]:
for word in ["king", "queen", "paris", "dog"]:
    word_id = word_to_id[word]
    before = model["initial_W_in"][word_id]
    after = model["W_in"][word_id]
    print(f"\n{word}")
    print("  before:    ", before)
    print("  after:     ", after)
    print("  difference:", after - before)
    print("  movement norm:", np.linalg.norm(after - before))


In [ ]:
pair_counts = Counter((id_to_word[int(center)], id_to_word[int(context)])
                      for center, context in training_pairs)
print("Most frequent observed ordered pairs:")
for pair, count in pair_counts.most_common(10):
    print(f"{pair!s:25} {count}")
for center_word in ["king", "dog", "paris"]:
    contexts = {context: count for (center, context), count in pair_counts.items() if center == center_word}
    total = sum(contexts.values())
    print(center_word, "empirical P(context | center):",
          {word: round(count / total, 3) for word, count in contexts.items()})


At fixed parameters, a pair appearing $r$ times contributes $r\nabla L_{co}$
to the **summed** gradient. In the averaged objective its weight is $r/N$.
SGD visits those observations separately, so their gradients change as the
parameters change; the actual trajectory is not simply a final vector multiplied
by frequency. Frequency also affects output vectors whenever a word is a context.


In [ ]:
frequency_center, frequency_context = word_to_id["dog"], word_to_id["cat"]
frequency_hidden, _, frequency_probabilities = forward(
    frequency_center, model["initial_W_in"], model["initial_W_out"])
single_pair_gradient = backward(frequency_center, frequency_context, frequency_hidden,
                                frequency_probabilities, model["initial_W_out"])[0][frequency_center]
repeat_count = pair_counts[("dog", "cat")]
print("One observation's center gradient:", single_pair_gradient)
print(f"{repeat_count} observations, frozen parameters:", repeat_count * single_pair_gradient)
print("Weight in the mean objective:", repeat_count / len(training_pairs))


Now measure the same selected input-vector cosine similarities at every saved epoch. Epoch zero is random initialization, not a neutral semantic baseline.


In [ ]:
similarity_history = {
    (first, second): np.array([cosine_similarity(snapshot[word_to_id[first]], snapshot[word_to_id[second]])
                              for snapshot in model["embedding_history"]])
    for first, second in SELECTED_WORD_PAIRS
}
checkpoints = sorted({epoch for epoch in [0, 10, 50, 100, EPOCHS] if epoch <= EPOCHS})
print("epoch " + " ".join(f"{a}/{b:7}" for a, b in SELECTED_WORD_PAIRS))
for epoch in checkpoints:
    print(f"{epoch:5d} " + " ".join(f"{similarity_history[pair][epoch]:+12.4f}" for pair in SELECTED_WORD_PAIRS))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for (first, second), values in similarity_history.items():
    ax.plot(values, label=f"{first} / {second}")
ax.set(xlabel="Completed epochs", ylabel="Cosine similarity", ylim=(-1.05, 1.05),
       title="How input-vector directions change during training")
ax.legend(loc="best")
fig.tight_layout()
plt.show()


Compare these curves with the loss curve. A decreasing prediction loss does not require every intuitively related pair's cosine to increase. The objective concerns center-to-context scores, and several input geometries can fit those scores.


**Think first:** Does repeating the entire corpus ten times change its empirical context probabilities?

<details>
<summary>Reveal the answer</summary>

No. All relative counts stay the same, so the mean objective is unchanged. With the same number of epochs, however, SGD performs ten times as many updates; this changes the training budget and potentially the result.

</details>


# 13. Effect of context window size

We change only the window radius: 1, 2, and 3. The vocabulary, initialization,
dimension, learning rate, and number of epochs stay fixed. The radius-2 model is
reused. Larger windows generally create more pairs, so equal epochs do **not**
mean equal numbers of updates. Once a window spans a sentence, enlarging it adds nothing.


In [ ]:
window_models = {WINDOW_SIZE: model}
window_pair_sets = {}
for radius in [1, 2, 3]:
    pairs_for_window = encode_training_pairs(sentences, radius)
    window_pair_sets[radius] = pairs_for_window
    if radius not in window_models:
        window_models[radius] = train_softmax(pairs_for_window, V, EMBEDDING_DIM,
                                              EPOCHS, LEARNING_RATE, SEED)

def print_model_comparison(models, pair_sets, setting_name):
    print(f"{setting_name:>8} {'pairs':>6} {'updates':>8} {'loss':>8} {'entropy':>8} "
          + " ".join(f"{a}/{b}" for a, b in SELECTED_WORD_PAIRS))
    for setting, result in sorted(models.items()):
        pairs = pair_sets[setting]
        scores = [cosine_similarity(result["W_in"][word_to_id[a]], result["W_in"][word_to_id[b]])
                  for a, b in SELECTED_WORD_PAIRS]
        print(f"{setting:8} {len(pairs):6} {len(pairs) * EPOCHS:8} {result['losses'][-1]:8.4f} "
              f"{empirical_context_entropy(pairs):8.4f} " + " ".join(f"{score:+.4f}" for score in scores))

print_model_comparison(window_models, window_pair_sets, "window")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, radius in zip(axes, [1, 2, 3]):
    plot_embedding_matrix(window_models[radius]["W_in"], ax, f"Window radius = {radius}")
fig.tight_layout()
plt.show()


Compare which pairs were added before interpreting the vectors. On large natural
corpora, narrower windows emphasize local grammatical relationships, while wider
windows can capture broader topical associations. These tiny short lines cannot
reliably demonstrate that distinction.

The labels and their empirical entropy change with the window, so raw final
losses are **different objectives**, not a contest with a universal winner.
Each PCA is fitted separately; its axes, orientation, and scale are not aligned
with other panels. Compare neighbors within a panel and cosine values in the
original space, not a word's absolute position across panels.


**Think first:** What changes when the window radius increases?

<details>
<summary>Reveal the answer</summary>

The set and number of context observations, their frequency distribution, the training objective, and usually the number of SGD updates per epoch. The vocabulary and embedding dimension need not change.

</details>


# 14. Effect of embedding dimension

Now fix the baseline pairs and vary $D\in\{2,5,10,20\}$. The model has
$VD+DV=2VD$ trainable scalars. Its pre-softmax score matrix is
$W_{in}W_{out}$ of shape `(V, V)`, whose rank is at most $\min(D,V)$.
Larger $D$ allows a richer factorization, but when $D>V$ the score-matrix rank
cannot increase further.

We use the same seed and initialization distribution, not identical matrices
across different shapes. This is a one-seed educational comparison, not a robust
statistical study of which dimension performs best.


In [ ]:
dimension_models = {EMBEDDING_DIM: model}
for dimension in [2, 5, 10, 20]:
    if dimension not in dimension_models:
        dimension_models[dimension] = train_softmax(training_pairs, V, dimension,
                                                    EPOCHS, LEARNING_RATE, SEED)
dimension_pair_sets = {dimension: training_pairs for dimension in dimension_models}
print_model_comparison(dimension_models, dimension_pair_sets, "D")
print("Parameter counts:", {dimension: 2 * V * dimension for dimension in [2, 5, 10, 20]})


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, dimension in zip(axes.flat, [2, 5, 10, 20]):
    plot_embedding_matrix(dimension_models[dimension]["W_in"], ax, f"Embedding dimension = {dimension}")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots()
for dimension in [2, 5, 10, 20]:
    ax.plot(dimension_models[dimension]["losses"], label=f"D={dimension}")
ax.set(xlabel="Completed epochs", ylabel="Mean cross-entropy", title="Capacity and optimization")
ax.legend()
plt.show()


More parameters can fit observed context probabilities more flexibly, but they
also leave more freedom for arbitrary geometry when evidence is scarce. Larger
vectors cost more memory and computation and can be sensitive to initialization
and learning rate. Lower training loss does not establish better semantic
generalization. PCA may hide more information at larger dimensions; check each
panel's retained-variance percentage.

**Try next:** change the seed and rerun the notebook. Which conclusions survive?
Keep the corpus and other settings fixed before attributing a difference to $D$.


# 15. The problem with full softmax

One pair's logits require `(D,) @ (D, V) → (V,)`: roughly $DV$
multiply-accumulate terms. Normalization costs another $O(V)$; the dense output
gradient also costs $O(DV)$. Thus forward/backward cost is **$O(VD)$ per pair**,
and the two parameter matrices store $2VD$ numbers.

The arithmetic below uses $D=100$ and float64 (8 bytes per parameter). It computes
counts only; we do not allocate matrices for the hypothetical vocabularies.


In [ ]:
illustrative_dimension = 100
print(f"{'V':>10} {'logit dot-product terms/pair':>29} {'two matrices (MB)':>20}")
for vocabulary_size in [10, 10_000, 1_000_000]:
    dot_product_terms = vocabulary_size * illustrative_dimension
    parameter_megabytes = 2 * vocabulary_size * illustrative_dimension * 8 / 1_000_000
    print(f"{vocabulary_size:10,} {dot_product_terms:29,} {parameter_megabytes:20,.3f}")


At one million words, even a single observed pair scores one million candidates.
The memory estimate covers only the parameters; dense gradients and other
intermediate arrays need additional memory. Float32 would halve parameter storage.

Two alternatives used in Word2Vec are:

| Method | Training task | Typical per-pair score computation |
|---|---|---|
| Full softmax | Normalize over every candidate word | $O(VD)$ |
| Negative sampling | Classify one observed pair and $K$ noise pairs | $O((K+1)D)$ |
| Hierarchical softmax | Predict decisions along a word's tree path | $O(D\times\text{path length})$, often described as $O(D\log V)$ |

Hierarchical softmax defines a normalized word distribution through a binary
tree. Path lengths depend on the tree, often a frequency-based Huffman tree;
they are not all exactly $\log_2 V$. We discuss it but do not implement it.
Negative sampling keeps two embedding tables but scores and updates only selected
words, avoiding a vocabulary-wide softmax. We build it next.


**Think first:** Does negative sampling eliminate the need to store embeddings for the whole vocabulary?

<details>
<summary>Reveal the answer</summary>

No. The embedding tables still require O(VD) storage. The main saving is scoring and updating a small set of context words for each observed pair.

</details>


# 16. Negative sampling

### Intuition: replace a multiclass task with binary tasks

Instead of asking “Which word in the entire vocabulary is the context?”, ask
“Does this center-context pair come from the observed corpus or from a noise
sampler?” For each observed **positive** pair, draw $K$ **negative** context words.

For example, `king → queen` is observed; `king → animal` might be generated as
noise. A sampled word is not necessarily semantically wrong: the sampler can
draw a word that is a real context in a different observation. Negative means
“assigned a noise label in this training step,” not “known to be unrelated.”

Let $v_c$ be the center input vector and $u_w$ the output/context vector, both
shape `(D,)`. The score $s=v_c\cdot u_w$ is a scalar and

$$\sigma(s)=\frac{1}{1+e^{-s}}$$

maps it into $(0,1)$. A large positive score favors an observed-pair label; a
large negative score favors a noise label. These binary scores do **not** form
a normalized probability distribution over context words.


In [ ]:
def sigmoid(scores):
    scores = np.asarray(scores, dtype=float)
    result = np.empty_like(scores)
    nonnegative = scores >= 0
    result[nonnegative] = 1.0 / (1.0 + np.exp(-scores[nonnegative]))
    exponentials = np.exp(scores[~nonnegative])
    result[~nonnegative] = exponentials / (1.0 + exponentials)
    return result

sigmoid_example = np.array([-1000., -2., 0., 2., 1000.])
print("Scores: ", sigmoid_example)
print("Sigmoid:", sigmoid(sigmoid_example))


### Objective and derivatives

For positive context $o$ and negative IDs $n_1,\ldots,n_K$, minimize

$$L_{NS}=-\log\sigma(v_c\cdot u_o)
-\sum_{k=1}^K\log\sigma(-v_c\cdot u_{n_k}).$$

For one binary label $t\in\{0,1\}$ with score $s$, binary cross-entropy is
$\ell=-t\log\sigma(s)-(1-t)\log(1-\sigma(s))$.
Using $\sigma'(s)=\sigma(s)(1-\sigma(s))$ gives

$$\frac{\partial\ell}{\partial s}
=-t(1-\sigma(s))+(1-t)\sigma(s)=\sigma(s)-t.$$

The positive derivative is $\sigma(s_o)-1$; each negative derivative is
$\sigma(s_{n_k})$. We sum, rather than average, the $K+1$ binary losses per
positive pair, so changing $K$ changes the loss scale and relative noise weight.

| Quantity | Shape | Operation |
|---|---|---|
| Selected output columns $U_S$ | `(D, K+1)` | Gather positive first, then negatives |
| Scores $s$ | `(K+1,)` | $v_c U_S$ |
| Labels $t$ | `(K+1,)` | `[1, 0, ..., 0]` |
| Score gradients $g$ | `(K+1,)` | $\sigma(s)-t$ |
| Center gradient | `(D,)` | $g U_S^T$ |
| Selected output gradients | `(D, K+1)` | $v_c^T g$ |

Repeated negative IDs represent repeated terms in the loss. Their gradients
must be **added into the same output column**, not overwritten.


To avoid taking logs of saturated sigmoid values, use
$-\log\sigma(s)=\log(1+e^{-s})$ and
$-\log\sigma(-s)=\log(1+e^s)$. NumPy's `logaddexp(0, s)` evaluates
$\log(1+e^s)$ stably. The following loss assumes the positive score is first.


In [ ]:
def negative_sampling_loss(scores):
    scores = np.asarray(scores, dtype=float)
    return float(np.logaddexp(0.0, -scores[0]) + np.logaddexp(0.0, scores[1:]).sum())

def negative_sampling_forward(center_id, sampled_ids, input_embeddings, output_weights):
    hidden = input_embeddings[center_id].copy()
    selected_outputs = output_weights[:, sampled_ids].copy()
    scores = hidden @ selected_outputs
    assert selected_outputs.shape == (hidden.size, len(sampled_ids))
    assert scores.shape == (len(sampled_ids),)
    return hidden, selected_outputs, scores

def negative_sampling_backward(hidden, selected_outputs, scores):
    gradient_scores = sigmoid(scores)
    gradient_scores[0] -= 1.0
    gradient_hidden = gradient_scores @ selected_outputs.T
    gradient_selected_outputs = np.outer(hidden, gradient_scores)
    return gradient_hidden, gradient_selected_outputs, gradient_scores


In [ ]:
ns_example_input = np.array([[0.2, -0.1], [0.0, 0.2], [-0.3, 0.1]])
ns_example_output = np.array([[0.1, -0.2, 0.3], [0.1, 0.1, 0.6]])
ns_example_ids = np.array([1, 2, 2])  # positive = 1; the same negative appears twice
ns_hidden, ns_selected, ns_scores = negative_sampling_forward(
    0, ns_example_ids, ns_example_input, ns_example_output)
ns_hidden_gradient, ns_output_gradient, ns_score_gradient = negative_sampling_backward(
    ns_hidden, ns_selected, ns_scores)
print("hidden / selected outputs shapes:", ns_hidden.shape, ns_selected.shape)
print("Scores:", ns_scores, "shape:", ns_scores.shape)
print("Sigmoid scores:", sigmoid(ns_scores))
print("Loss:", negative_sampling_loss(ns_scores))
print("Score gradients:", ns_score_gradient)
print("Center gradient:", ns_hidden_gradient)
print("Per-occurrence output gradients:\n", ns_output_gradient)


In this example the scores are `[-0.05, 0, 0]`. At a zero negative score,
$\sigma(0)=0.5$, so each negative contributes loss $\log 2$ and score gradient
$0.5$. Output word 2 occurs twice, so its total gradient is twice one occurrence's
gradient. The following accumulation uses `np.add.at`: ordinary advanced-index
`+=` can fail to accumulate repeated indices correctly.


In [ ]:
ns_accumulated_output = np.zeros_like(ns_example_output)
np.add.at(ns_accumulated_output.T, ns_example_ids, ns_output_gradient.T)
print("Output gradients after accumulating repeated IDs:\n", ns_accumulated_output)
np.testing.assert_allclose(ns_accumulated_output[:, 2], ns_output_gradient[:, 1] + ns_output_gradient[:, 2])


**Numerical gradient check, with repeated negatives:** keep the sampled IDs fixed while perturbing parameters. Resampling inside the loss would compare different random objectives and invalidate a finite-difference check.


In [ ]:
ns_check_input, ns_check_output = initialize_parameters(4, 2, seed=19)
ns_check_ids = np.array([2, 1, 1, 3])
ns_check_hidden, ns_check_selected, ns_check_scores = negative_sampling_forward(
    1, ns_check_ids, ns_check_input, ns_check_output)
ns_check_grad_hidden, ns_check_grad_selected, _ = negative_sampling_backward(
    ns_check_hidden, ns_check_selected, ns_check_scores)
ns_analytic_input = np.zeros_like(ns_check_input)
ns_analytic_input[1] = ns_check_grad_hidden
ns_analytic_output = np.zeros_like(ns_check_output)
np.add.at(ns_analytic_output.T, ns_check_ids, ns_check_grad_selected.T)

def check_negative_loss():
    return negative_sampling_loss(negative_sampling_forward(1, ns_check_ids, ns_check_input, ns_check_output)[2])

for parameter, analytic in [(ns_check_input, ns_analytic_input), (ns_check_output, ns_analytic_output)]:
    numerical = finite_difference_gradient(check_negative_loss, parameter)
    print("Maximum absolute SGNS gradient error:", np.max(np.abs(analytic - numerical)))
    np.testing.assert_allclose(analytic, numerical, rtol=1e-5, atol=1e-7)


### Drawing negative words

Let $f(w)$ be a token's frequency in the original corpus, not its pair count.
We use the noise distribution

$$q(w)=\frac{f(w)^{0.75}}{\sum_j f(j)^{0.75}}.$$

The exponent flattens the distribution compared with raw frequency: rare words
get relatively more weight. For counts `[1, 16]`, the powered counts are `[1, 8]`;
the normalized noise probabilities are `[1/9, 8/9]` rather than `[1/17, 16/17]`.

We precompute its cumulative distribution (CDF), shape `(V,)`, once. Each uniform
draw chooses a CDF interval. For this teaching version we reject the **current
positive context**, resampling until we have $K$ negatives. Thus the actual
negative distribution for positive $o$ is $q(w)/(1-q(o))$ for $w\ne o$.
We allow repeated negatives and do not exclude the center or all other true contexts.


In [ ]:
token_counts = Counter(word for sentence in sentences for word in sentence)
noise_weights = np.array([token_counts[word] ** 0.75 for word in vocabulary], dtype=float)
noise_probabilities = noise_weights / noise_weights.sum()
noise_cdf = np.cumsum(noise_probabilities)
noise_cdf[-1] = 1.0
NEGATIVE_SAMPLES = 5
print("word       count   noise probability")
for word, probability in zip(vocabulary, noise_probabilities):
    print(f"{word:10} {token_counts[word]:5d} {probability:19.4f}")
assert np.isclose(noise_probabilities.sum(), 1.0)


In [ ]:
def sample_negatives(positive_context_id, count, cumulative_probabilities, rng):
    positive_mass = cumulative_probabilities[positive_context_id]
    if positive_context_id > 0:
        positive_mass -= cumulative_probabilities[positive_context_id - 1]
    if positive_mass >= 1.0:
        raise ValueError("Cannot sample a different context from this distribution")
    negative_ids = []
    while len(negative_ids) < count:
        draws = rng.random(count - len(negative_ids))
        candidates = np.searchsorted(cumulative_probabilities, draws, side="right")
        negative_ids.extend(candidates[candidates != positive_context_id].tolist())
    return np.asarray(negative_ids, dtype=np.int64)

negative_example = sample_negatives(word_to_id["queen"], NEGATIVE_SAMPLES, noise_cdf,
                                   np.random.default_rng(SEED))
print("Positive: king → queen")
print("Sampled negatives:", [("king", id_to_word[int(index)]) for index in negative_example])
assert word_to_id["queen"] not in negative_example


The CDF avoids rebuilding a vocabulary-sized probability array for each pair.
Binary searches add $O(K\log V)$ sampling work, with extra draws for rejected
positives. Rejection can be slow if one word holds nearly all probability mass.
Efficient production samplers use other precomputed data structures; our
$O((K+1)D)$ statement concerns the score and gradient calculations.

### Train with sparse updates

To make the loss curve comparable over time, create one **fixed set of evaluation
negatives** per training pair. Training draws fresh negatives at each update.
The fixed evaluation set measures this same corpus with fixed noise; it is not
a held-out semantic benchmark or an exact expectation over every possible noise draw.


In [ ]:
def make_evaluation_samples(pairs, negative_count, cumulative_probabilities, seed):
    rng = np.random.default_rng(seed)
    return np.array([np.concatenate(([context], sample_negatives(context, negative_count,
                                                                cumulative_probabilities, rng)))
                     for _, context in pairs])

def mean_negative_sampling_loss(pairs, evaluation_samples, input_embeddings, output_weights):
    losses = [negative_sampling_loss(negative_sampling_forward(center, sampled_ids,
                                                               input_embeddings, output_weights)[2])
              for (center, _), sampled_ids in zip(pairs, evaluation_samples)]
    return float(np.mean(losses))


In [ ]:
def negative_sampling_epoch(pairs, input_embeddings, output_weights, negative_count,
                            cumulative_probabilities, learning_rate, shuffle_rng, noise_rng):
    for center, context in pairs[shuffle_rng.permutation(len(pairs))]:
        negatives = sample_negatives(context, negative_count, cumulative_probabilities, noise_rng)
        sampled_ids = np.concatenate(([context], negatives))
        hidden, selected_outputs, scores = negative_sampling_forward(
            center, sampled_ids, input_embeddings, output_weights)
        gradient_hidden, gradient_selected, _ = negative_sampling_backward(hidden, selected_outputs, scores)
        input_embeddings[center] -= learning_rate * gradient_hidden
        np.add.at(output_weights.T, sampled_ids, -learning_rate * gradient_selected.T)


Only the center input row and sampled output columns are updated. The selected columns and hidden vector are copied during the forward pass, and both gradients are calculated before either parameter update. No dense `(D, V)` gradient is allocated in this training step.


In [ ]:
def train_negative_sampling(pairs, vocabulary_size, cumulative_probabilities,
                            embedding_dim=5, epochs=300, learning_rate=0.05,
                            negative_count=5, seed=SEED):
    input_embeddings, output_weights = initialize_parameters(vocabulary_size, embedding_dim, seed)
    shuffle_rng, noise_rng = np.random.default_rng(seed + 1), np.random.default_rng(seed + 2)
    evaluation_samples = make_evaluation_samples(pairs, negative_count, cumulative_probabilities, seed + 3)
    losses = [mean_negative_sampling_loss(pairs, evaluation_samples, input_embeddings, output_weights)]
    start_time = perf_counter()
    for epoch in range(epochs):
        negative_sampling_epoch(pairs, input_embeddings, output_weights, negative_count,
                                cumulative_probabilities, learning_rate, shuffle_rng, noise_rng)
        losses.append(mean_negative_sampling_loss(pairs, evaluation_samples, input_embeddings, output_weights))
    return {"W_in": input_embeddings, "W_out": output_weights, "losses": np.array(losses),
            "evaluation_samples": evaluation_samples, "elapsed_seconds": perf_counter() - start_time}


In [ ]:
negative_model = train_negative_sampling(training_pairs, V, noise_cdf, EMBEDDING_DIM,
                                         EPOCHS, LEARNING_RATE, NEGATIVE_SAMPLES, SEED)
print("Initial / final fixed-noise loss:", negative_model["losses"][[0, -1]])
print("Zero-score baseline (K+1) log(2):", (NEGATIVE_SAMPLES + 1) * np.log(2))
assert np.isfinite(negative_model["W_in"]).all() and np.isfinite(negative_model["W_out"]).all()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(model["losses"])
axes[0].set(title="Full softmax", xlabel="Epoch", ylabel="Mean multiclass cross-entropy")
axes[1].plot(negative_model["losses"], color="tab:orange")
axes[1].set(title="Negative sampling (fixed evaluation noise)", xlabel="Epoch",
            ylabel="Mean sum of K+1 binary losses")
fig.tight_layout()
plt.show()


### Compare the two models

| Property | Full softmax | Negative sampling |
|---|---|---|
| Supervision per observed pair | One categorical context label | One positive plus $K$ noise labels |
| Output | Normalized $P(o\mid c)$ over vocabulary | Binary data-versus-noise scores for selected pairs |
| Score/gradient work | $O(VD)$ | $O((K+1)D)$, plus sampling |
| Output columns updated | All | Only sampled columns |
| Loss scale | One multiclass loss | Sum of $K+1$ binary losses |

We keep data, seed, dimension, epoch count, and learning rate matched to make the
comparison inspectable. The objectives and gradient scales still differ; this
is not separately tuned benchmarking. Do not rank the methods by their raw losses.


In [ ]:
print(f"{'pair':20} {'softmax cosine':>15} {'SGNS cosine':>15}")
for first, second in SELECTED_WORD_PAIRS:
    center_index, other_index = word_to_id[first], word_to_id[second]
    softmax_cosine = cosine_similarity(W_in[center_index], W_in[other_index])
    negative_cosine = cosine_similarity(negative_model["W_in"][center_index], negative_model["W_in"][other_index])
    print(f"{first + '/' + second:20} {softmax_cosine:15.4f} {negative_cosine:15.4f}")
for label, result in [("Full softmax", model), ("Negative sampling", negative_model)]:
    print(label, "king neighbors:", most_similar_in_matrix("king", result["W_in"]))
    print(f"  Measured training + evaluation time: {result['elapsed_seconds']:.2f} seconds")
print("Score dot-product terms per pair:",
      {"full softmax": V * EMBEDDING_DIM, "negative sampling": (NEGATIVE_SAMPLES + 1) * EMBEDDING_DIM})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_embedding_matrix(W_in, axes[0], "Full softmax")
plot_embedding_matrix(negative_model["W_in"], axes[1], "Negative sampling")
fig.tight_layout()
plt.show()


On a tiny vocabulary, Python loop overhead, sampling, and evaluation can outweigh
the arithmetic savings: SGNS need not be faster here. These timings include
evaluation and the full-softmax model's snapshot copies; they are illustrative,
not a production benchmark. Both methods retain two vocabulary-wide parameter tables.

Different neighbors are plausible because the objectives differ. Inspect shared
contexts and the original-space similarities; separately fitted PCA pictures
cannot establish that one method is better.

**What does negative sampling approximate?** Fresh samples provide a Monte Carlo
estimate of an expected **noise-classification loss**. SGNS is a computationally
cheap alternative training objective for learning word vectors; it is not an
unbiased approximation of the full-softmax gradient or its denominator. Our
positive-exclusion rule further specifies which noise expectation we estimate.


**Think first:** Should the sigmoid scores for every possible context sum to one?

<details>
<summary>Reveal the answer</summary>

No. Each sigmoid belongs to a binary classification task. There is no vocabulary-wide normalization in SGNS, so these scores are not P(context | center) from the full-softmax model.

</details>


# 17. Relationship to the real Word2Vec algorithm

Mikolov and colleagues introduced the widely used Word2Vec architectures and
their efficient training techniques in these original papers:

- [Efficient Estimation of Word Representations in Vector Space (2013)](https://arxiv.org/abs/1301.3781).
- [Distributed Representations of Words and Phrases and their Compositionality (2013)](https://arxiv.org/abs/1310.4546).

Our notebook exposes the learning mechanics. Practical implementations add choices
needed for large data and fast training:

| Practical feature | Why it matters | What we do here |
|---|---|---|
| Large corpora | Many varied contexts supply evidence for linguistic relationships | Nine short sentences; no semantic benchmark |
| Frequent-word subsampling | Randomly discard many common-token occurrences to reduce their dominance and work | Keep every occurrence |
| Negative sampling / hierarchical softmax | Avoid scoring every vocabulary word for each observation | Implement SGNS; explain the tree alternative |
| Efficient operations | Compiled loops, sparse updates, vectorized numerical kernels, and parallelism improve throughput | Readable Python loops and NumPy operations |
| Vocabulary pruning | Drop very rare words to limit memory and unreliable estimates | Retain every word |
| Learning-rate schedules | Reduce update sizes as training progresses | Fixed learning rate |
| Randomized window radius | Vary effective context distance and pair weighting | Fixed radius, all in-window positions equally weighted |

Subsampling frequent words is **not** the same as drawing negative samples:
subsampling changes which token occurrences supply positive training contexts;
negative sampling constructs noise-classification examples.

Word2Vec learns a static vector per vocabulary item. It cannot give `bank`
different input vectors in “river bank” and “bank loan,” and our simple version
cannot produce an embedding for an unseen word. Modern contextual models solve
a different representation problem; they are outside this notebook's scope.


**Think first:** Why might vocabulary pruning improve practicality but remove useful information?

<details>
<summary>Reveal the answer</summary>

It reduces memory and work by excluding rare words whose vectors have little evidence, but it also removes those words and their contexts. The cutoff is a data and application tradeoff.

</details>


# 18. Compare with a library implementation (optional)

Only now do we use a library implementation. The following cell is **disabled
by default** and the rest of the notebook does not depend on it. If you want to
run it, install `gensim` in your notebook environment from a terminal, then set
`RUN_GENSIM = True`. Follow the [official Gensim Word2Vec documentation](https://radimrehurek.com/gensim/models/word2vec.html)
for installation compatibility and parameter details.

| Parameter | Connection to our implementation |
|---|---|
| `vector_size` | Embedding dimension $D$ |
| `window` | Maximum context radius |
| `sg=1` | Skip-Gram; `sg=0` selects CBOW |
| `negative=5`, `hs=0` | Five noise samples, hierarchical softmax off |
| `ns_exponent=0.75` | Powered frequency distribution for noise |
| `min_count=1` | Keep every vocabulary word |
| `sample=0` | Disable frequent-word subsampling for this small example |
| `shrink_windows=False` | Use a fixed effective window radius |
| `alpha`, `min_alpha` | Initial and minimum learning rates; equal values keep them fixed |
| `epochs`, `seed` | Training passes and random initialization setting |
| `workers=1` | Avoid parallel update-order variability |

Even with similar settings, expect different numbers: initialization, ordering,
sampling details, optimized update code, and precision differ. A single worker
helps reproducibility, but Gensim also documents Python hash randomization as a
factor across processes; setting only `seed` is not a universal exact-match guarantee.


In [ ]:
RUN_GENSIM = False

if RUN_GENSIM:
    try:
        from gensim.models import Word2Vec
    except ImportError as error:
        print("Optional Gensim comparison unavailable. Check the environment and Gensim installation.")
        print(error)
    else:
        library_model = Word2Vec(
            sentences=sentences, vector_size=EMBEDDING_DIM, window=WINDOW_SIZE,
            sg=1, negative=NEGATIVE_SAMPLES, hs=0, ns_exponent=0.75,
            min_count=1, sample=0, shrink_windows=False,
            alpha=LEARNING_RATE, min_alpha=LEARNING_RATE,
            epochs=EPOCHS, seed=SEED, workers=1,
        )
        print("Gensim king embedding:", library_model.wv["king"])
        print("Gensim king neighbors:", library_model.wv.most_similar("king", topn=5))
else:
    print("Optional Gensim comparison skipped; all core results use NumPy.")


# 19. Concept checks

Questions throughout the notebook have focused on individual steps. Now connect
the steps without looking at the code. Open each answer only after explaining it
in your own words.


**Think first:** Why does one-hot multiplication act as a row lookup, and what changes if the input is not one-hot?

<details>
<summary>Reveal the answer</summary>

The product is a weighted sum of matrix rows. One-hot weights leave exactly one row. With several nonzero weights it instead combines multiple rows; for example, averaged context one-hots can produce an averaged context embedding.

</details>


**Think first:** For one full-softmax training pair, which parameters receive gradients?

<details>
<summary>Reveal the answer</summary>

Only the center row of W_in receives an input gradient. Generally every column of W_out receives an output gradient because every candidate participates in softmax normalization. With SGNS only the selected output columns receive gradients.

</details>


**Think first:** Why can a lower training loss coexist with a lower king–queen cosine similarity?

<details>
<summary>Reveal the answer</summary>

The loss rewards prediction of observed contexts using input-to-output dot products. It does not constrain king's and queen's input vectors to align. Shared contexts can induce alignment, but direct co-occurrence and low loss alone do not guarantee it.

</details>


**Think first:** What does negative sampling approximate, and what does it not approximate?

<details>
<summary>Reveal the answer</summary>

Random negatives estimate an expected noise-classification objective. They do not compute the full-softmax distribution or an unbiased estimate of its gradient. The chosen noise distribution and number of negatives help define the learning problem.

</details>


**Think first:** Why might antonyms be close in a word embedding space?

<details>
<summary>Reveal the answer</summary>

Words such as hot and cold can fill similar sentence positions and share context words. Distributional similarity captures usage patterns and linguistic roles, not just synonymy.

</details>


**Think first:** How would you make the window experiment more convincing?

<details>
<summary>Reveal the answer</summary>

Use more varied text, compare multiple seeds, report the number of updates and objective changes, and evaluate a specific linguistic task on held-out evidence. Equal epochs alone do not control training work when pair counts differ.

</details>


# 20. Final summary

```text
Text corpus
    ↓
Tokenization
    ↓
Vocabulary
    ↓
(center, context) training pairs
    ↓
One-hot / word index
    ↓
Embedding lookup
    ↓
Predict context (full softmax) / distinguish observed and noise pairs (SGNS)
    ↓
Calculate loss
    ↓
Backpropagation
    ↓
Update embedding matrices
    ↓
Learned word vectors
    ↓
Cosine similarity / downstream NLP tasks
```

**Main insight:** Skip-Gram never receives an instruction saying which words have
similar meanings. Words appearing in similar contexts face similar prediction
problems and receive related gradient signals through shared output vectors.
Useful semantic structure can emerge from those updates. It depends on the data
and training choices; similarity is not explicitly guaranteed by the objective.

You have implemented both objectives with NumPy, traced their derivatives,
checked the gradients numerically, and built experiments that expose how the
representations change.

# Things I should be able to explain after completing this notebook

1. What exactly is a Skip-Gram training example?
2. What is stored inside the embedding matrix?
3. Why does one-hot multiplication act as an embedding lookup?
4. How does the model learn which words are related?
5. What role does cross-entropy play?
6. How are the embedding vectors updated during backpropagation?
7. Why is cosine similarity useful for embeddings?
8. Why is full softmax inefficient?
9. How does negative sampling address that cost, and how does its objective differ?
10. How do window size and embedding dimension affect the learned representation?

**A final exercise:** explain one `king → queen` update using only a diagram,
the shapes of the two matrices, and the sign of each prediction error. Then
explain what changes when you replace full softmax with five negative samples.
